> ### 🎓 강사 가이드 — Analytics & Reporting Demo 개요
>
> | 항목 | 내용 |
> |------|------|
> | ⏱️ 소요 시간 | 전체 약 75분 |
> | 🎯 핵심 목표 | 이력 데이터 집계 → 트렌드 분석 → 자동 보고서 생성 전체 파이프라인 이해 |
> | 🔑 핵심 개념 | 분석 도구(analytics_tools) 6개 함수 활용 |
>
> **💡 강조 포인트**
> - 이 데모는 AI 에이전트가 경영진 보고 자료를 자동 생성하는 시나리오입니다
> - "데이터 → 숫자 → LLM 해석 → 한국어 보고서" 흐름이 핵심입니다
>
> **❓ 생각해보기**: "매달 작성하는 설비 현황 보고서를 AI가 자동으로 생성하면 얼마나 시간이 절감될까요?"


# 04. Analytics & Reporting Demo

**목적**: `analytics_tools.py`의 모든 기능을 단계별로 시연합니다.

- 이력 조회 + 통계 분석
- 트렌드 분석 + 24h 예측
- 의사결정 (정비 권고)
- 마크다운 보고서 자동 생성
- 에이전트 시나리오 (질문 → 분석 → 보고서)

In [ ]:
# 섹션 1: 설정 및 임포트
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '../tools'))
sys.path.insert(0, os.path.join(os.getcwd(), '../data'))

from analytics_tools import (
    get_signal_history,
    get_signal_statistics,
    get_trend_analysis,
    compare_machines,
    get_maintenance_decision,
    generate_equipment_health_report,
    generate_quality_report,
    generate_factory_weekly_summary,
    search_maintenance_log,
)

print('analytics_tools 임포트 완료')

> ### 🎓 강사 가이드 — 섹션 2: 이력 데이터 생성
>
> | 항목 | 내용 |
> |------|------|
> | ⏱️ 소요 시간 | 약 5분 |
> | 🎯 핵심 목표 | 30일치 시뮬레이션 데이터 생성 원리 이해 |
> | 🔑 핵심 개념 | 시계열 데이터, 정상/이상 구간 혼합 |
>
> **💡 강조 포인트**
> - 실제 현장에서는 MES, SCADA 시스템에서 데이터를 가져옵니다
> - 이 실습에서는 `generate_historical_data()`로 현실적인 패턴의 시뮬레이션 데이터를 생성합니다


---
## 섹션 2: 30일 이력 데이터 생성

In [ ]:
# generate_demo_history.py 실행으로 JSONL 이력 데이터 생성
import subprocess
import sys

script_path = os.path.join(os.getcwd(), '../data/generate_demo_history.py')
result = subprocess.run([sys.executable, script_path], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('[STDERR]', result.stderr[:300])

> ### 🎓 강사 가이드 — 섹션 3: 이력 조회 및 통계
>
> | 항목 | 내용 |
> |------|------|
> | ⏱️ 소요 시간 | 약 15분 |
> | 🎯 핵심 목표 | `query_historical_data()` Tool로 집계 통계 조회 |
> | 🔑 핵심 개념 | 평균, 최대, 이상 발생 횟수, 가동률 등 KPI 계산 |
>
> **💡 강조 포인트**
> - KPI(핵심성과지표)를 정의하는 것이 분석의 첫 단계입니다
> - 같은 데이터도 어떤 KPI를 보느냐에 따라 해석이 달라집니다
>
> **❓ 생각해보기**: "설비 관리자와 경영진이 각각 원하는 KPI는 어떻게 다를까요?"


---
## 섹션 3: 이력 조회 및 통계

In [ ]:
import json

# 최근 48개 이력 조회 (anomaly)
history = get_signal_history('M001', 'anomaly', last_n=48)
print(f'조회 결과: {history["summary"]}')
print(f'총 레코드 수: {history["count"]}')
print(f'\n[최근 5개 레코드]')
for r in history['records'][-5:]:
    print(f'  {r["ts"][:16]}  score={r["value"]:.4f}  status={r["status"]}')

In [ ]:
# 7일 통계 분석
stats = get_signal_statistics('M001', 'anomaly', period_hours=168)
print(stats['summary'])
print('\n[통계 상세]')
for k, v in stats['statistics'].items():
    print(f'  {k:20s}: {v}')

> ### 🎓 강사 가이드 — 섹션 4: 트렌드 분석 및 시각화
>
> | 항목 | 내용 |
> |------|------|
> | ⏱️ 소요 시간 | 약 15분 |
> | 🎯 핵심 목표 | 시계열 트렌드 분석 + 24시간 예측 시각화 |
> | 🔑 핵심 개념 | 트렌드(상승/하락/안정), 이동평균, 선형 예측 |
>
> **💡 강조 포인트**
> - 트렌드 방향만 봐도 설비 상태의 변화를 예측할 수 있습니다
> - 예측이 100% 맞을 필요는 없습니다 — 방향성(악화 중인가?)이 중요합니다
>
> **❓ 생각해보기**: "예측이 틀렸을 때 현장에서 어떻게 대응해야 할까요?"


---
## 섹션 4: 트렌드 분석 + 시각화

In [ ]:
import matplotlib
matplotlib.use('Agg')  # 비대화형 백엔드
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import numpy as np

# 7일 트렌드 분석
trend = get_trend_analysis('M001', 'anomaly_score', period_days=7)
print(trend['summary'])
print(f"  추세 기울기: {trend['trend']['slope']:+.6f}")
print(f"  변화율: {trend['trend']['change_pct']:+.2f}%")
print(f"  내일 예측: {trend['forecast']['forecast_24h']}")

# 일별 데이터 출력
print('\n[일별 요약]')
for dv in trend['daily_values']:
    bar = '█' * int(dv['mean'] * 20)
    alert_mark = ' ⚠️' if dv['alert_count'] > 0 else ''
    print(f"  {dv['date']}  {dv['mean']:.3f} {bar}{alert_mark}")

In [ ]:
# 이상점수 추이 그래프 (7일)
history_plot = get_signal_history('M001', 'anomaly', last_n=7*6)
records = history_plot['records']

if records:
    dates = [datetime.fromisoformat(r['ts']) for r in records]
    values = [r['value'] for r in records]
    statuses = [r['status'] for r in records]

    fig, ax = plt.subplots(figsize=(12, 5))

    # 정상/이상 구분 색상
    normal_idx = [i for i, s in enumerate(statuses) if s == 'NORMAL']
    anomaly_idx = [i for i, s in enumerate(statuses) if s == 'ANOMALY']

    ax.plot(dates, values, color='#3498db', linewidth=1.2, alpha=0.8, label='이상점수')

    # 이상 포인트 강조
    if anomaly_idx:
        ax.scatter(
            [dates[i] for i in anomaly_idx],
            [values[i] for i in anomaly_idx],
            color='#e74c3c', s=40, zorder=5, label='ANOMALY 경보'
        )

    # 임계값 선
    ax.axhline(y=0.75, color='#e74c3c', linestyle='--', linewidth=1.5,
               alpha=0.7, label='임계값 (0.75)')

    # 트렌드 선 (선형 회귀)
    if len(values) >= 2:
        x_num = np.arange(len(values))
        coeffs = np.polyfit(x_num, values, 1)
        trend_line = np.polyval(coeffs, x_num)
        ax.plot(dates, trend_line, color='#f39c12', linewidth=2,
                linestyle='-', alpha=0.8, label='추세선')

    ax.set_title(f'M001 이상탐지 점수 추이 (최근 7일)', fontsize=13, fontweight='bold')
    ax.set_xlabel('날짜')
    ax.set_ylabel('이상점수')
    ax.set_ylim(0, 1.05)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig('../outputs/anomaly_trend_7d.png', dpi=120, bbox_inches='tight')
    plt.close()
    print('그래프 저장: ../outputs/anomaly_trend_7d.png')
else:
    print('데이터 없음')

> ### 🎓 강사 가이드 — 섹션 5: 의사결정 및 비용 분석
>
> | 항목 | 내용 |
> |------|------|
> | ⏱️ 소요 시간 | 약 15분 |
> | 🎯 핵심 목표 | AI 의사결정 지원 — 정비 비용 vs 고장 손실 비교 분석 |
> | 🔑 핵심 개념 | TCO(총소유비용), 예방비용 vs 고장손실 |
>
> **💡 강조 포인트**
> - "정비 비용 < 고장 손실"이면 예방정비가 경제적입니다
> - 이 분석이 경영진 보고에서 가장 설득력 있는 부분입니다
>
> **❓ 생각해보기**: "불확실성(예: 고장 확률 30~70%)을 비용 분석에 어떻게 반영할 수 있을까요?"


---
## 섹션 5: 의사결정 + 비용 분석 시각화

In [ ]:
# 의사결정 결과
decision = get_maintenance_decision('M001')
print(f'\n{'='*50}')
print(f'설비 M001 의사결정 결과')
print(f'{'='*50}')
print(f'결정    : {decision["decision"]}')
print(f'긴급도  : {decision["urgency_score"]}/100')
print(f'권장일  : {decision["recommended_date"]}')
print(f'\n[결정 근거]')
for r in decision['reasoning']:
    print(f'  • {r}')
print(f'\n[권장 조치]')
for i, a in enumerate(decision['action_items']):
    print(f'  {i+1}. {a}')
print(f'\n[비용 분석]')
ca = decision['cost_analysis']
print(f'  계획 정비: {ca["planned_cost"]:,}원')
print(f'  긴급 정비: {ca["emergency_cost"]:,}원')
print(f'  절감 가능: {ca["saving_by_planning"]:,}원')
print(f'\n[방치 시 리스크]')
print(f'  {decision["risk_if_ignored"]}')

In [ ]:
# 비용 분석 바 차트
ca = decision['cost_analysis']

fig, ax = plt.subplots(figsize=(7, 4))

categories = ['긴급 정비\n(방치 시)', '계획 정비\n(권장)']
values_cost = [ca['emergency_cost'], ca['planned_cost']]
colors = ['#e74c3c', '#2ecc71']

bars = ax.bar(categories, values_cost, color=colors, width=0.5, edgecolor='white')

# 절감액 표시
ax.annotate(
    f'절감 {ca["saving_by_planning"]:,}원',
    xy=(1, ca['planned_cost']),
    xytext=(1.3, (ca['emergency_cost'] + ca['planned_cost']) / 2),
    arrowprops=dict(arrowstyle='->', color='gray'),
    fontsize=10, color='#2c3e50'
)

# 값 라벨
for bar, val in zip(bars, values_cost):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 30000,
        f'{val:,}원',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_title('정비 비용 비교', fontsize=13, fontweight='bold')
ax.set_ylabel('비용 (원)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_ylim(0, ca['emergency_cost'] * 1.3)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/maintenance_cost_comparison.png', dpi=120, bbox_inches='tight')
plt.close()
print('비용 비교 차트 저장: ../outputs/maintenance_cost_comparison.png')

> ### 🎓 강사 가이드 — 섹션 6: 자동 보고서 생성
>
> | 항목 | 내용 |
> |------|------|
> | ⏱️ 소요 시간 | 약 15분 |
> | 🎯 핵심 목표 | LLM으로 데이터를 한국어 경영 보고서로 자동 변환 |
> | 🔑 핵심 개념 | 데이터 → 프롬프트 → LLM → 한국어 보고서 |
>
> **💡 강조 포인트**
> - 숫자 데이터를 프롬프트에 삽입하면 LLM이 경영진용 언어로 해석합니다
> - 보고서 형식(요약, 현황, 위험, 권고)을 프롬프트에 지정해야 일관성이 유지됩니다
>
> **🚨 주의사항**: LLM이 생성한 보고서는 반드시 전문가가 검토 후 배포하세요 (hallucination 위험)


---
## 섹션 6: 자동 보고서 생성

In [ ]:
# 설비 건강도 보고서 생성 및 출력
from IPython.display import Markdown, display

health_report = generate_equipment_health_report('M001', period_days=7)
print(f'[보고서 생성 완료 — 총 {len(health_report.split(chr(10)))}줄]\n')

# 파일 저장
report_path = '../outputs/equipment_health_report_M001.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(health_report)
print(f'저장: {report_path}\n')

# 렌더링 (Jupyter에서)
display(Markdown(health_report))

In [ ]:
# 품질 보고서 생성
quality_report = generate_quality_report('LINE-A', period_days=7)
print(f'[품질 보고서 — 총 {len(quality_report.split(chr(10)))}줄]\n')

qr_path = '../outputs/quality_report_LINE_A.md'
with open(qr_path, 'w', encoding='utf-8') as f:
    f.write(quality_report)
print(f'저장: {qr_path}\n')

display(Markdown(quality_report))

> ### 🎓 강사 가이드 — 섹션 7: 에이전트 시나리오
>
> | 항목 | 내용 |
> |------|------|
> | ⏱️ 소요 시간 | 약 15분 |
> | 🎯 핵심 목표 | 분석 도구 체인을 에이전트가 자동으로 실행하는 전체 시나리오 이해 |
> | 🔑 핵심 개념 | 에이전트가 질문 → 도구 선택 → 실행 → 종합 보고서 생성 |
>
> **💡 강조 포인트**
> - 이 시나리오가 X1 에이전트의 최종 사용 형태입니다
> - 경영진이 질문하면 에이전트가 데이터를 조회하고 보고서를 자동 생성합니다
>
> **❓ 생각해보기**: "이 에이전트를 Slack, Teams 등 메신저에 연결하면 어떤 사용법이 가능할까요?"


---
## 섹션 7: 에이전트 시나리오

**질문**: "M001 설비 지난 7일간 어떤 상태였나요? 정비가 필요한가요?"

아래는 에이전트가 이 질문을 받았을 때 자동으로 실행하는 도구 체인을 시뮬레이션합니다.

In [ ]:
print('=' * 60)
print('에이전트 시나리오: M001 설비 지난 7일 상태 분석')
print('=' * 60)

print('\n[Step 1] get_signal_statistics — 7일 통계 분석')
stats_result = get_signal_statistics('M001', 'anomaly', period_hours=168)
print(f'  → {stats_result["summary"]}')
print(f'  → 평균 이상점수: {stats_result["statistics"]["mean"]:.3f}')
print(f'  → 경보 횟수: {stats_result["statistics"]["alert_count"]}회')
print(f'  → 추세: {stats_result["statistics"]["trend_direction"]}')

print('\n[Step 2] get_trend_analysis — 트렌드 및 예측')
trend_result = get_trend_analysis('M001', 'anomaly_score', period_days=7)
print(f'  → {trend_result["summary"]}')
print(f'  → 리스크 궤적: {trend_result["trend"]["risk_trajectory"]}')
print(f'  → 내일 예측값: {trend_result["forecast"]["forecast_24h"]}')

print('\n[Step 3] get_maintenance_decision — 종합 의사결정')
decision_result = get_maintenance_decision('M001')
print(f'  → {decision_result["summary"]}')
print(f'  → 결정: {decision_result["decision"]}')
print(f'  → 권장 정비일: {decision_result["recommended_date"]}')

print('\n[Step 4] generate_equipment_health_report — 보고서 생성')
final_report = generate_equipment_health_report('M001', period_days=7)
print(f'  → 보고서 생성 완료 ({len(final_report)}자)')

print('\n[에이전트 최종 답변]')
print('-' * 50)
anomaly_mean = stats_result['statistics']['mean']
alert_cnt = stats_result['statistics']['alert_count']
trajectory = trend_result['trend']['risk_trajectory']
dec_text = decision_result['decision']
urgency = decision_result['urgency_score']

response = f"""
M001 설비의 지난 7일간 상태 분석 결과입니다:

📊 **상태 요약**
- 이상탐지 평균 점수: {anomaly_mean:.3f} (임계값 0.75)
- 7일 경보 발생: {alert_cnt}회
- 추세: {'악화 중 ↗️' if trajectory == 'escalating' else ('개선 중 ↘️' if trajectory == 'recovering' else '안정 →')}

🔧 **정비 권고**
- 결정: **{dec_text}**
- 긴급도: {urgency}/100
- 권장 정비일: {decision_result['recommended_date']}

📋 **주요 근거**
{chr(10).join('- ' + r for r in decision_result['reasoning'])}

💰 **비용 정보**
- 계획 정비 비용: {decision_result['cost_analysis']['planned_cost']:,}원
- 긴급 정비 대비 절감: {decision_result['cost_analysis']['saving_by_planning']:,}원

상세 보고서는 아래 섹션을 확인하세요.
"""
print(response)

In [ ]:
# 최종 보고서 렌더링
display(Markdown(final_report))

# 공장 주간 요약도 출력
weekly = generate_factory_weekly_summary()
weekly_path = '../outputs/factory_weekly_summary.md'
with open(weekly_path, 'w', encoding='utf-8') as f:
    f.write(weekly)
print(f'\n공장 주간 요약 저장: {weekly_path}')
display(Markdown(weekly))

In [ ]:
# 유지보수 로그 검색 시연
print('=== 유사 사례 검색: 베어링 과열 ===\n')
search_result = search_maintenance_log('베어링 과열', machine_id=None, top_k=3)
print(search_result['summary'])
for r in search_result['results']:
    print(f"\n[{r['date']}] {r['machine_id']}")
    print(f"  증상: {r['description']}")
    print(f"  조치: {r['action_taken']}")
    print(f"  결과: {r['result']}")

In [ ]:
# 설비 비교
print('=== 설비 비교 (M001 vs M002 vs M003) ===\n')
cmp = compare_machines(['M001', 'M002', 'M003'], 'anomaly_score', 24)
print(cmp['summary'])
print()
for r in cmp['ranking']:
    icon = {'CRITICAL': '🔴', 'WARNING': '🟠', 'NORMAL': '🟢'}.get(r['status'], '')
    print(f"  [{r['rank']}위] {r['machine_id']} — {icon} {r['status']} | 평균={r['mean']:.3f} | 경보={r['alert_count']}회 | 추세={r['trend_direction']}")

---

## 📝 과제 — S4 Lab 4: Analytics & Reporting

### 기본 과제 (필수)
1. **7일 불량률 트렌드**: 지난 7일간 불량률 데이터를 추출하여 꺾은선 그래프로 시각화하고, 트렌드 방향(상승/하락/안정)을 분석하세요
2. **자동 보고서 검토**: `generate_report()` 함수로 생성된 보고서를 읽고, 개선이 필요한 부분(과장된 표현, 근거 없는 주장 등)을 3가지 이상 찾아 수정 의견을 작성하세요

### 심화 과제 (선택)
1. **PDF 보고서 자동 생성**: `reportlab` 또는 `fpdf2` 라이브러리를 사용하여 분석 결과와 차트를 포함한 PDF 보고서를 자동 생성하는 코드를 작성하세요
   - 힌트: `pip install reportlab` 후 `canvas.Canvas()` 사용
2. **이메일 자동 발송 시뮬레이션**: `smtplib`를 사용하여 보고서를 이메일로 발송하는 함수를 작성하세요 (실제 발송 대신 dry-run 모드로 구현해도 됩니다)

### 제출 기준
- [ ] 모든 셀 오류 없이 실행 완료
- [ ] 7일 불량률 트렌드 시각화 차트 포함
- [ ] 자동 보고서 검토 의견 3가지 이상
- [ ] (심화) PDF 보고서 생성 코드 또는 이메일 발송 시뮬레이션 코드 포함
